# Training con Time-Domain features 

Este NoteBook se centra en entrenar clasificadores con las características de dominio temporal de los datos, a diferencia de los otros, que entrenamos con datos crudos. 
El objetivo es comparar clasificadores con FE vs datos crudos, en aspectos como:
- Accuracy
- Latencia
- Tiempo de training
- Robustez al ruido

Aplicaremos cross validation para obtener unos promedios más confiables.

In [ ]:
import pandas as pd
import numpy as np
import time
from td_features import td_features_dataset
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)

## Cargamos Datos

In [ ]:
df = pd.read_csv("dataset.csv") # dataset completo
y = df.iloc[:, -1].values
df = df.drop(df.columns[-1], axis=1) # eliminamos label

## Pasamos a características del dominio temporal

In [3]:
df = td_features_dataset(df, scaler=StandardScaler())
X = df.values

## Splits

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

## Entrenando Clasificadores 

### SVM

In [ ]:
param_grid = [
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    },
    {
        "kernel": ["poly"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"],
        "degree": [2, 3, 4],
        "coef0": [0.0, 1.0],
    },
]

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

start = time.time()
grid.fit(X_train, y_train)
end = time.time()
best_svm = grid.best_estimator_

print("Mejor estimador:", grid.best_estimator_)
print("Mejores parámetros:", grid.best_params_)
print("Mejor score CV:", grid.best_score_)
print(f"Tiempo de training: {end-start}:.2f s")

Fitting 5 folds for each of 68 candidates, totalling 340 fits
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   7.9s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   8.8s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   8.9s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   9.2s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   9.9s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=  10.4s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=  10.4s
[CV] END .....................C=0.1, gamma=0.001, kernel=rbf; total time=  10.9s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=  11.6s
[CV] END .....................C=0.1, gamma=0.001, kernel=rbf; total time=  13.9s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=  16.1s
[CV] END .....................C=0.1, gamma=scal

### Random Forest

In [ ]:
param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_features':     ['sqrt', 'log2', None],
    'max_depth':        [100, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

start = time.time()
grid_rf.fit(X_train, y_train)
best_rf_model = grid_rf.best_estimator_
end = time.time()

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Mejor CV score:     {grid_rf.best_score_:.4f}")
print(f"Test score:         {grid_rf.best_estimator_.score(X_test, y_test):.4f}")
print(f"Tiempo de training: {end-start}:.2f s")

Mejores parámetros: {'max_depth': 100, 'max_features': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Mejor CV score:     0.5436
Test score:         0.5531


### AdaBoost

In [ ]:
param_grid = {
    'n_estimators':  [25, 50, 75, 100],
    'learning_rate': [0.01, 0.1, 0.5, 0.9],
    'estimator__max_depth' : [1, 2, 3, 5]
}

grid_ada = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

start = time.time()
grid_ada.fit(X_train, y_train)
end = time.time()
best_ada_model = grid_ada.best_estimator_

print(f"Mejores parámetros: {grid_ada.best_params_}")
print(f"Mejor CV score:     {grid_ada.best_score_:.4f}")
print(f"Test score:         {grid_ada.best_estimator_.score(X_test, y_test):.4f}")
print(f"Tiempo de training: {end-start}:.2f s")

Mejores parámetros: {'estimator__max_depth': 5, 'learning_rate': 0.1, 'n_estimators': 100}
Mejor CV score:     0.5260
Test score:         0.5486


## Ruido y Latencia

In [ ]:
noise = np.random.uniform(-0.3*np.std(X_test), 0.3*np.std(X_test), X_test.shape)
X_noise = X_test + noise

In [ ]:
def accuracy_and_latency(model, X_test, y_test):
    start = time.time()
    y_pred = model.predict(X_test)
    end = time.time()
    elapsed = end - start
    acc = accuracy_score(y_test, y_pred)
    return [acc, elapsed]

acc_svm, time_svm = accuracy_and_latency(best_svm, X_noise, y_test)
acc_rf, time_rf = accuracy_and_latency(best_rf_model, X_noise, y_test)
acc_ada, time_ada = accuracy_and_latency(best_ada_model, X_noise, y_test)

print(f"SVM - Accuracy: {acc_svm:.4f}, Tiempo: {time_svm:.4f}s")
print(f"Random Forest - Accuracy: {acc_rf:.4f}, Tiempo: {time_rf:.4f}s")
print(f"AdaBoost - Accuracy: {acc_ada:.4f}, Tiempo: {time_ada:.4f}s")

array([[1.33447516, 0.91643894, 2.90329146],
       [0.20726318, 1.78533397, 1.84919794],
       [0.76988709, 2.35069232, 1.58312512]])

In [ ]:
# hacer tabla comparativa accuracy con vs sin noise